In [6]:
# Kaggle Dataset Direct Download
!pip install -q kaggle
!curl -L -o chest-xray.zip https://www.kaggle.com/api/v1/datasets/download/paultimothymooney/chest-xray-pneumonia
!unzip -q chest-xray.zip -d chest_xray_data
print("✅ Real Chest X-Ray Dataset Downloaded & Ready!")

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 2349M  100 2349M    0     0   126M      0  0:00:18  0:00:18 --:--:--  103M
✅ Real Chest X-Ray Dataset Downloaded & Ready!


In [7]:
# ==============================================================================
# Project: Chest X-Ray Medical Image Classification
# Architecture: Fine-Tuned ResNet18 (Transfer Learning)
# ==============================================================================

import os
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

# 1. GPU Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executing on device: {device}")

# 2. Preprocessing Pipeline (Automatic 224x224 Resizing)
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 3. Load Real Kaggle X-Ray Dataset
data_dir = 'chest_xray_data/chest_xray/train'
train_dataset = ImageFolder(root=data_dir, transform=data_transforms)

# Fast DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
print(f"Loaded {len(train_dataset)} Real Chest X-Ray Images!")
print(f"Classes Detected: {train_dataset.classes}")

# 4. ResNet18 Model Setup
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)  # Normal (0) vs Pneumonia (1)
model = model.to(device)

# 5. Loss & Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

# 6. Fast Training Loop (Quick Verification)
print("\n[INFO] Starting Fast Model Fine-Tuning on GPU...")
model.train()

for batch_idx, (images, labels) in enumerate(train_loader):
    images, labels = images.to(device), labels.to(device)

    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    if batch_idx % 20 == 0:
        print(f"Batch [{batch_idx}/{len(train_loader)}] - Loss: {loss.item():.4f}")

    # Fast stop after 50 batches for quick execution
    if batch_idx >= 50:
        break

# 7. Save Model
torch.save(model.state_dict(), "medical_resnet18_real.pth")
print("\n✅ Success: Real Chest X-Ray Model Fine-Tuned & Saved Successfully!")

Executing on device: cuda
Loaded 5216 Real Chest X-Ray Images!
Classes Detected: ['NORMAL', 'PNEUMONIA']

[INFO] Starting Fast Model Fine-Tuning on GPU...
Batch [0/163] - Loss: 0.7542
Batch [20/163] - Loss: 0.1631
Batch [40/163] - Loss: 0.0802

✅ Success: Real Chest X-Ray Model Fine-Tuned & Saved Successfully!
